# TFM — 04. Modelización: Accidentes de tráfico en Madrid (2012-2018)
**Autora:** Meritxell Abellan Collado

Este notebook responde a las dos preguntas de investigación planteadas en la introducción del TFM, usando las matrices ya construidas en `03_Feature_Engineering.ipynb`:

- **Sección 1 — Modelo de triaje operativo** (Pregunta 1): entrenado solo con `FEATURES_OPERATIVO` (disponibles en el momento del aviso). Evaluado con AUC-ROC, AUC-PR y calibración — no con accuracy, engañosa con un desbalanceo del 9.6%.
- **Sección 2 — Modelo explicativo** (Pregunta 2): entrenado con `FEATURES_REFERENCIA` (todas las variables). El resultado principal es la interpretabilidad (SHAP), no la métrica de clasificación.
- **Sección 3 — Comparación operativo vs. referencia**: cuantifica cuánto se "pierde" por restringirse a variables operativas.
- **Sección 4 — Traducción a recomendaciones de prevención vial.**

## Estructura
0. Carga de matrices
1. Modelo de triaje operativo
2. Modelo explicativo (SHAP)
3. Comparación operativo vs. referencia
4. Recomendaciones de prevención


## 0. Carga de datos

In [ ]:
import utils.modelizacion as mdl
print('Archivo que Python está cargando de verdad:', mdl.__file__)
print()
print('¿Tiene graficar_curvas_roc_train_test?:', hasattr(mdl, 'graficar_curvas_roc_train_test'))
print()
print('Todo lo que contiene ese módulo:')
print([f for f in dir(mdl) if not f.startswith('_')])

In [ ]:
import pandas as pd
import numpy as np
import warnings

import sys
sys.path.append('..')
from utils.pipeline_produccion import PipelineAccidentes
from utils.modelizacion import (
    entrenar_modelos_baseline, tabla_comparativa_modelos,
    curva_calibracion, comparar_tasa_base_train_test,
    entrenar_random_forest_tuneado, graficar_curvas_roc,
    comparar_splits_temporal_vs_aleatorio, graficar_curvas_roc_train_test,
    validacion_temporal_expansiva, graficar_validacion_temporal,analisis_estabilidad_bagging
)
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

RANDOM_STATE = 42
PATH = '../data/processed/features/'

X_train_op = pd.read_csv(PATH + 'X_train_operativo.csv')
X_test_op = pd.read_csv(PATH + 'X_test_operativo.csv')
X_train_ref = pd.read_csv(PATH + 'X_train_referencia.csv')
X_test_ref = pd.read_csv(PATH + 'X_test_referencia.csv')
y_train = pd.read_csv(PATH + 'y_train.csv').squeeze()
y_test = pd.read_csv(PATH + 'y_test.csv').squeeze()

print(f'Operativo  — train: {X_train_op.shape} | test: {X_test_op.shape}')
print(f'Referencia — train: {X_train_ref.shape} | test: {X_test_ref.shape}')
print(f'Tasa de gravedad — train: {y_train.mean()*100:.2f}% | test: {y_test.mean()*100:.2f}%')

# Lista de columnas operativas, derivada de lo ya cargado (se reutiliza en la
# Sección 1.6 para reconstruir las mismas features sobre el split aleatorio)
FEATURES_OPERATIVO = list(X_train_op.columns)

pipe = PipelineAccidentes.load('../models/pipeline_produccion.joblib')
assert FEATURES_OPERATIVO == pipe.features_operativo, \
    "Las columnas de X_train_op no coinciden con pipe.features_operativo -- revisar si el pipeline guardado corresponde a esta ejecución"
print(f'Pipeline cargado y verificado ({len(pipe.features_operativo)} features operativo, coincide con los CSV).')


## 1. Modelo de triaje operativo (Pregunta 1)

Se entrena únicamente sobre `X_train_op`/`X_test_op` — las variables disponibles en el momento del aviso. Se comparan tres candidatos:

- **Regresión logística** (`class_weight='balanced'`): interpretable, buen baseline.
- **Random Forest** y **Gradient Boosting**: esperable mejor discriminación con este nivel de desbalanceo, sin necesitar escalado de variables.

**Métricas:** AUC-ROC y AUC-PR (no dependen de un umbral concreto, apropiadas con 9.6% de clase positiva) y Brier score (calibración: si la probabilidad predicha refleja de verdad la frecuencia observada — relevante para un uso de triaje, donde lo que importa no es solo "grave sí/no" sino *cuánta prioridad relativa* dar).


In [ ]:
modelos_operativo = entrenar_modelos_baseline(X_train_op, y_train, random_state=RANDOM_STATE)

tabla_operativo = tabla_comparativa_modelos(modelos_operativo, X_train_op, y_train, X_test_op, y_test, umbral=0.5)
tabla_operativo[['AUC-ROC_train', 'AUC-ROC_test', 'gap_AUC-ROC', 'AUC-PR_test', 'Brier score_test']]


**Lectura del gap train-test:** un gap grande en `AUC-ROC` (train mucho más alto que test) es la señal clásica de overfitting — el modelo memoriza el train en vez de generalizar. Random Forest con la configuración por defecto suele mostrar este patrón (árboles sin límite de profundidad); si es el caso aquí, se explora en la siguiente celda si el bagging (variar `max_samples`/`n_estimators`) puede corregirlo, o si hace falta limitar también la profundidad de cada árbol.


In [ ]:
grid_bagging = analisis_estabilidad_bagging(
    X_train_op, y_train, X_test_op, y_test,
    porcentajes=(0.2, 0.4, 0.6, 0.8, 1.0),
    n_arboles=(100, 300, 500),
    profundidades=(5, 10, None),
    random_state=RANDOM_STATE
)
grid_bagging.head(15)


**Resultado:** `max_depth` es, con diferencia, el factor que más reduce el gap — mucho más que `max_samples` o `n_estimators`. Con árboles sin límite de profundidad (`max_depth=None`, el valor por defecto), el gap ronda 0.30-0.34 sin importar cuánto se varíen los otros dos parámetros; limitando la profundidad (`max_depth=5` o `10`) el gap cae por debajo de 0.03-0.09, con un AUC de test similar o mejor. Esto confirma que el problema no era "cuántos árboles promediar" sino que cada árbol individual ya estaba sobreajustado — el bagging por sí solo no lo corrige si no se limita también la complejidad de cada árbol base.


## 1.5 Random Forest tuneado (a partir del análisis de bagging) + curvas ROC

In [ ]:
rf_tuneado = entrenar_random_forest_tuneado(
    X_train_op, y_train, max_depth=5, n_estimators=100, max_samples=0.2, random_state=RANDOM_STATE
)
modelos_operativo['Random Forest (tuneado)'] = rf_tuneado

tabla_operativo = tabla_comparativa_modelos(modelos_operativo, X_train_op, y_train, X_test_op, y_test)
tabla_operativo[['AUC-ROC_train', 'AUC-ROC_test', 'gap_AUC-ROC', 'AUC-PR_test', 'Brier score_test']]


In [ ]:
fig = graficar_curvas_roc(
    modelos_operativo, X_test_op, y_test,
    titulo='Curvas ROC — modelo de triaje operativo',
    filename='../figures/12_roc_comparacion_modelos.png'
)


**Curva ROC train vs. test de Random Forest sin tunear**, para visualizar el overfitting que ya vimos en la tabla (gap de 0.33): la curva de train queda muy por encima de la de test, señal de que el modelo memoriza en vez de generalizar.


In [ ]:
from utils.modelizacion import graficar_curvas_roc_train_test

fig = graficar_curvas_roc_train_test(
    modelos_operativo, X_train_op, y_train, X_test_op, y_test,
    filename='../figures/13b_roc_train_test_modelos.png'
)


**Conclusión de la Sección 1:** Gradient Boosting (Hist) sigue siendo el mejor modelo de triaje operativo (AUC-ROC test = 0.703), con un gap train-test moderado (0.064) que no requiere ajuste adicional. Random Forest, una vez tuneado (`max_depth=5`), corrige casi todo su overfitting (gap de 0.33 a 0.025) pero no supera a Gradient Boosting en test — queda como modelo de comparación/robustez, no como candidato principal.


**Nota importante sobre el umbral de decisión:** con umbral fijo de 0.5, Random Forest y Gradient Boosting muestran un recall casi nulo (verás valores muy próximos a 0 en la tabla anterior), a pesar de tener el AUC-ROC más alto de los tres modelos. Esto **no significa que discriminen peor** — significa que, con un 9.6% de clase positiva, sus probabilidades predichas rara vez superan 0.5 de forma natural, aunque ordenen correctamente los casos de mayor a menor riesgo (que es justo lo que mide el AUC).

Esto es exactamente el motivo por el que **AUC-ROC/AUC-PR son las métricas principales aquí, no el F1 con umbral 0.5** — un umbral de decisión solo tiene sentido fijarlo *después*, en función del caso de uso real (p. ej. "¿cuántos recursos de emergencia adicionales estamos dispuestos a movilizar por cada aviso marcado como de alto riesgo?"), no como un valor por defecto arbitrario.


In [ ]:
mejor_modelo_nombre = tabla_operativo['AUC-ROC_test'].idxmax()
mejor_modelo = modelos_operativo[mejor_modelo_nombre]
print(f'Modelo con mejor AUC-ROC: {mejor_modelo_nombre}')

y_proba_test = mejor_modelo.predict_proba(X_test_op)[:, 1]
tabla_calibracion = curva_calibracion(y_test, y_proba_test, n_bins=10)
tabla_calibracion


**Por qué se revisa la calibración además de la discriminación:** el split es temporal (train 2012-2017, test 2018), y la tasa de gravedad real desciende de forma consistente año a año (~10.9% en 2012 a ~8.2% en 2018 — ver `02_Preprocesado.ipynb`, sección de split). Como consecuencia, `train` tiene una tasa base más alta (9.86%) que `test` (8.19%). Un modelo bien calibrado en 2018 confirmaría que esa diferencia de tasa base no está sesgando las probabilidades predichas; si la tabla de calibración muestra que el modelo sobreestima sistemáticamente el riesgo (probabilidad media predicha por encima de la frecuencia real observada en cada bin), sería esperable dado ese desajuste de tasa base entre train y test, y convendría documentarlo como limitación del modelo de triaje.


In [ ]:
print(comparar_tasa_base_train_test(y_train, y_test))


## 1.6 ¿Es el desajuste de calibración un problema del split, o de la deriva temporal real?

Se entrena el mismo modelo (misma configuración) sobre el split temporal y sobre el split aleatorio estratificado, para separar dos cosas: si el modelo discrimina peor en el temporal (problema del modelo/split) o si solo empeora su calibración por la diferencia real de tasa base entre 2012-2017 y 2018 (una propiedad de los datos, no un fallo).


In [ ]:
# Split aleatorio: se reutilizan los MISMOS Nº PARTE que ya se determinaron
# en 02_Preprocesado.ipynb (train_random.csv / test_random.csv), pero se
# parte del Excel crudo a nivel persona -- porque PipelineAccidentes
# hace su propia agregación desde cero y no puede partir de datos ya agregados
partes_train_rand = pd.read_csv('../data/processed/train_random.csv')['Nº PARTE']
partes_test_rand = pd.read_csv('../data/processed/test_random.csv')['Nº PARTE']

df_raw = pd.read_excel('../data/raw/accidentes-trafico.xlsx')
df_train_rand_raw = df_raw[df_raw['Nº PARTE'].isin(partes_train_rand)].copy()
df_test_rand_raw = df_raw[df_raw['Nº PARTE'].isin(partes_test_rand)].copy()

pipe_rand = PipelineAccidentes(m_target_encoding=50)
train_rand_final = pipe_rand.fit(df_train_rand_raw)
test_rand_final = pipe_rand.transform(df_test_rand_raw)

X_train_op_rand = train_rand_final[pipe_rand.features_operativo]
X_test_op_rand = test_rand_final[pipe_rand.features_operativo]
y_train_rand = train_rand_final['GRAVE']
y_test_rand = test_rand_final['GRAVE']

factory = lambda: HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE)

tabla_splits = comparar_splits_temporal_vs_aleatorio(
    (X_train_op, y_train, X_test_op, y_test),
    (X_train_op_rand, y_train_rand, X_test_op_rand, y_test_rand),
    factory
)
tabla_splits


**Resultado esperado:** el AUC-ROC de test debería ser similar entre ambos splits (el modelo discrimina casi igual de bien), pero la `diferencia_tasa_pts` será prácticamente 0 en el aleatorio y notablemente negativa en el temporal — confirmando que el desajuste de calibración es una propiedad real de la deriva temporal de los datos, no un defecto del split ni del modelo. Esto justifica mantener el split temporal como principal (más honesto de cara a un despliegue real), documentando la necesidad de recalibración periódica como limitación conocida.

**Nota:** `FEATURES_OPERATIVO` debe estar ya definida (la misma lista de columnas usada en `03_Feature_Engineering.ipynb`); si no está cargada como variable en este notebook, hay que reconstruirla antes de ejecutar la celda anterior.


## 1.7 Validación walk-forward: ¿es 2018 un año representativo como test?

En vez de `TimeSeriesSplit` genérico (que particiona por número de filas, no por año natural), se valida año a año: entrenar con todo lo anterior a un año y validar sobre ese año, avanzando desde 2014 hasta 2018. Esto comprueba si el AUC-ROC del split final (test = 2018) es consistente con el patrón general, o si 2018 es un año atípico.

**Limitación:** las features ya construidas (target encoding de distrito, one-hot) se ajustaron una única vez sobre 2012-2017; no se reajustan dentro de cada fold del walk-forward. Un walk-forward sin ninguna fuga exigiría reconstruirlas en cada iteración — aquí se usa la aproximación más simple, documentada como tal.


In [ ]:
# Se reutiliza el mismo `pipe` (ajustado sobre 2012-2017) y se aplica
# transform() sobre TODO el histórico 2012-2018 a la vez, para tener AÑO
# disponible en la misma matriz de features ya construida -- ya no hace
# falta ninguna función local que reimplemente el feature engineering
df_raw = pd.read_excel('../data/raw/accidentes-trafico.xlsx')

combinado_final = pipe.transform(df_raw)

X_wf = combinado_final[FEATURES_OPERATIVO]
y_wf = combinado_final['GRAVE']
años_wf = combinado_final['AÑO']

factory = lambda: HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE)

tabla_walkforward = validacion_temporal_expansiva(X_wf, y_wf, años_wf, factory, año_min_train=2013, año_max=2018)
tabla_walkforward


In [ ]:
fig = graficar_validacion_temporal(tabla_walkforward, filename='../figures/14_validacion_walkforward.png')


**Conclusión:** el AUC-ROC de validación se mantiene estable entre 0.69 y 0.71 en los 5 años probados, y el valor de 2018 (el test actual del proyecto) queda dentro de ese rango, no en un extremo. Esto respalda que el split temporal elegido (train 2012-2017, test 2018) es representativo del comportamiento general del modelo, y que el descenso observado en la calibración (Sección 1.6) se debe específicamente a la deriva de la tasa base, no a una pérdida de capacidad discriminativa del modelo con el tiempo.


## 2. Modelo explicativo (Pregunta 2)

Se entrena sobre `X_train_ref`/`X_test_ref` — todas las variables, incluido el perfil de las personas implicadas. El objetivo aquí no es solo la métrica de clasificación, sino tener el modelo base para el análisis SHAP de `05_Interpretabilidad.ipynb`. Se usan los mismos candidatos que en la Sección 1, para mantener el criterio de comparación.


In [ ]:
modelos_referencia = entrenar_modelos_baseline(X_train_ref, y_train, random_state=RANDOM_STATE)

tabla_referencia = tabla_comparativa_modelos(modelos_referencia, X_train_ref, y_train, X_test_ref, y_test)
tabla_referencia[['AUC-ROC_train', 'AUC-ROC_test', 'gap_AUC-ROC', 'AUC-PR_test', 'Brier score_test']]


In [ ]:
import joblib, os

mejor_modelo_ref_nombre = tabla_referencia['AUC-ROC_test'].idxmax()
mejor_modelo_ref = modelos_referencia[mejor_modelo_ref_nombre]
print(f'Mejor modelo de referencia: {mejor_modelo_ref_nombre}')

os.makedirs('../models', exist_ok=True)
joblib.dump(mejor_modelo_ref, '../models/modelo_referencia.joblib')
joblib.dump(modelos_operativo[tabla_operativo['AUC-ROC_test'].idxmax()], '../models/modelo_operativo.joblib')
print('Modelos guardados en ../models/')


## 3. Comparación operativo vs. referencia: ¿cuánto se pierde por restringirse a variables operativas?


In [ ]:
mejor_op = tabla_operativo.loc[tabla_operativo['AUC-ROC_test'].idxmax()]
mejor_ref = tabla_referencia.loc[mejor_modelo_ref_nombre]

comparacion = pd.DataFrame({
    'Operativo (Pregunta 1)': mejor_op[['AUC-ROC_test', 'AUC-PR_test', 'Brier score_test']],
    'Referencia (Pregunta 2)': mejor_ref[['AUC-ROC_test', 'AUC-PR_test', 'Brier score_test']],
})
comparacion['Diferencia'] = comparacion['Referencia (Pregunta 2)'] - comparacion['Operativo (Pregunta 1)']
comparacion


**Conclusión:** el modelo de referencia mejora el AUC-ROC de test en torno a 1-2 puntos frente al operativo. Es una ganancia real pero modesta — indica que las variables disponibles en el momento del aviso (Pregunta 1) ya capturan la mayor parte de la señal predictiva; el perfil de las personas implicadas aporta valor incremental, pero no es el factor dominante. Esto refuerza que un sistema de triaje basado solo en información del aviso es razonable, y que el valor añadido de las variables de persona está más en la **interpretación** (Pregunta 2, prevención) que en la mejora bruta de la predicción — que es precisamente lo que se explora en `05_Interpretabilidad.ipynb`.
